In [ ]:
# ==============================================================================
# ⚠️ IMPORTANT NOTICE: RUNTIME TYPE CONFIGURATION
# ==============================================================================
# Please ensure your Google Colab session is configured to run on a T4 (GPU) 
# hardware accelerator. 
#
# Running the local Ollama LLM model requires around 8-9 GB of VRAM to execute 
# efficiently. Running on a standard CPU runtime will take a very long time 
# or fail to start.
# ==============================================================================

## 01. Mount your Google Drive

In [ ]:
# ==============================================================================
# ## 01. Mount your Google Drive
# ==============================================================================
# This step securely connects your Google Colab instance to your personal 
# Google Drive. 
#
# Large Language Model (LLM) weights are massive (approx. 8-9 GB). 
#   - On the very first run: The model is downloaded from the web and saved to 
#     your Google Drive.
#   - On future runs: Colab will quickly copy the model locally from your Drive, 
#     saving you from redownloading it over the internet.
#
# Zero Data Charges:
#   Downloading the model from Ollama to Colab/Google Drive is a direct server-to-
#   server transfer. This does NOT use your local internet data package or count 
#   against your personal bandwidth plan.
#
# Requirements:
#   1. You will need roughly 8-9 GB of free storage space in your Google Drive.
#   2. Ensure that your Google Drive and this Colab notebook are accessed 
#      under the same Google account.
# ==============================================================================

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

## 02. Clone SmartBEM repo

In [ ]:
# ==============================================================================
# ## 02. Clone SmartBEM Repository
# ==============================================================================
# This step clones the SmartBEM-Studio repository to the local, 
# Colab runtime and configures the environment paths.
#
#   1. Creates a persistent `SmartBEM-Studio` cache directory on your Google Drive.
#   2. Clears any old repository folder inside the ephemeral `/content` path.
#   3. Clones the latest code repository from GitHub.
#   4. Copies the EnergyPlus Python utility files into the backend server directory.
#   5. Configures the active Python system path
# ==============================================================================

In [ ]:
# Set working directory and clone repository dynamically

import os, sys, shutil, json

# 1. Establish persistent folder on Google Drive for caching
drive_hvac_dir = "/content/drive/MyDrive/SmartBEM-Studio"
os.makedirs(drive_hvac_dir, exist_ok=True)

# 2. Establish local repository runtime folder
local_repo_path = "/content/SmartBEM-Studio"
%cd /content
if os.path.exists(local_repo_path):
    shutil.rmtree(local_repo_path)

print("Cloning repository from GitHub...")
!git clone https://github.com/kethshen/SmartBEM-Studio.git {local_repo_path}

colab_dir = os.path.join(local_repo_path, "backend_server")

# 3. Ensure eplus utility is copied for imports
eplus_src = os.path.join(local_repo_path, "EnergyPlus utility")
eplus_dest = os.path.join(colab_dir, "eplus")
if not os.path.exists(eplus_dest):
    shutil.copytree(eplus_src, eplus_dest)

%cd {colab_dir}
sys.path.append(colab_dir)
print(f"Working directory set to: {os.getcwd()}")

## 03. Load Ngrok Authtoken

In [ ]:
# ==============================================================================
# ## 03. Load Ngrok Authtoken
# ==============================================================================
# This use Ngrok to tunnel the Colab backend server to the internet. To do this, 
# you need a free Ngrok account.
#
# Steps to configure:
#   1. Create a free account at https://ngrok.com
#   2. Copy your Authtoken from your Ngrok dashboard.
#   3. Click the key icon (🔑 Secrets) in the left sidebar of Google Colab.
#   4. Add a new secret with Name = "NGROK_AUTHTOKEN" and Value = [Your Token].
#   5. Turn ON the "Notebook access" toggle for this secret.
#   6. Run this cell to load the token.
#
# Note:
#   Secrets are securely stored inside your Google Account and are only visible 
#   to you and notebooks you explicitly authorize.
# ==============================================================================

In [ ]:
# 4. Load Ngrok Authtoken from Google Colab Secrets
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')

    # Save it to local secrets.json so fastapi_server.py can parse it
    local_secrets = os.path.join(colab_dir, "secrets.json")
    with open(local_secrets, "w") as f:
        json.dump({"NGROK_AUTHTOKEN": ngrok_token}, f, indent=4)
    print("✅ Successfully loaded NGROK_AUTHTOKEN from Colab Secrets!")
except Exception as e:
    print("⚠️ Colab Secret 'NGROK_AUTHTOKEN' not found or failed to load.")
    print("Please click the key icon (🔑) in the Colab sidebar, add a secret named 'NGROK_AUTHTOKEN', grant access, and rerun.")

## 04. Install OpenStudio SDK


In [ ]:
# ==============================================================================
# ## 04. Install OpenStudio SDK
# ==============================================================================
# OpenStudio SDK is a comprehensive software development kit created by the 
# National Renewable Energy Laboratory (NREL) and its open-source developer 
# community (website: https://openstudio.net/ | repository: https://github.com/NREL/OpenStudio) 
# for building energy modeling.
#
# Instead of manually formatting raw EnergyPlus text files, this SDK is used to:
#     - Handle geometry calculations (defining zone layouts, walls, windows, doors).
#     - Match thermal boundaries between adjacent zones.
#     - Perform HVAC sizing calculations.
# ==============================================================================

In [ ]:
# Run this in a new cell at the top of your notebook:
from eplus import prepare_colab_eplus
prepare_colab_eplus()

## 05. Install Ngrok and FastAPI

In [ ]:
# ==============================================================================
# ## 05. Install Ngrok and FastAPI
# ==============================================================================
# This step installs the required network utilities to expose this colab backend.
#
#   - FastAPI: acts as the backend API server that receives plain-English requests, 
#     triggers simulations, and feeds results back to the frontend.
#   - Ngrok: Since the Colab runtime is firewalled without a public IP, Ngrok 
#     tunnels the local FastAPI port (8000) to a public URL so the local 
#     browser frontend can send requests to it.
# ==============================================================================

In [ ]:
!pip install pyngrok nest-asyncio fastapi uvicorn pydantic


## 06. Install Ollama local server

In [ ]:
# ==============================================================================
# ## 06. Install Ollama Local Server
# ==============================================================================
# Ollama allows running large language models (LLMs) locally. 
# This runs on the Colab GPU runtime to handle natural language processing.
#
#   - Installs system packages required for running GPU utilities.
#   - Downloads and installs the Ollama Linux engine.
#   - Installs the Python package for communicating with the Ollama server.
# ==============================================================================

In [ ]:
# 1. Install missing dependencies & GPU utils
!sudo apt-get install -y zstd
!sudo apt update && sudo apt install pciutils lshw -y

In [ ]:
# 2. Install Ollama Linux Engine
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Install Python SDK
!pip install ollama

## 07. [RUN ONCE] Pull Ollama Models from Ollama Library

In [ ]:
# ==============================================================================
# ## 07. [RUN ONCE] Pull Ollama Models from Ollama Library
# ==============================================================================
# IMPORTANT: Run the cells in this section ONLY if this is your first time 
# setting up the project, or if you need to download a new model.
# 
# This step may take some time to complete around 8-10 minutes.
#
#   - Configures the storage directory to point to your Google Drive cache folder.
#   - Connects to Ollama's public library and downloads the Gemma 3 model weights.
#   - Saves the downloaded weights directly to your Google Drive.
#
# Once downloaded, you can skip running these cells in future sessions, as we 
# will copy the cached model directly from your Drive.
# ==============================================================================

In [ ]:
import os
os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/SmartBEM-Studio/ollama_models/'


In [ ]:
!ollama pull gemma3:12b


In [ ]:
!ollama list

### Optional -  To remove a model use below

In [ ]:
# ==============================================================================
# ### Optional - Remove Ollama Model
# ==============================================================================
# This cell is a helper to delete local or cached models from your 
# Google Drive or local storage in case you want to free up space.
# You may have to delete the model weights in the Drive manually.
# ==============================================================================

In [ ]:
import subprocess
# Remove gemma manifest
!rm -rf "'/content/drive/MyDrive/SmartBEM-Studio/ollama_models/manifests/registry.ollama.ai/library/gemma3"


In [ ]:
# Easiest: just run ollama rm while OLLAMA_MODELS points to Drive
import os
os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/SmartBEM-Studio/ollama_models"
!ollama rm gemma3:4b


## 08. Copy Ollama model from Drive to Colab

In [ ]:
# ==============================================================================
# ## 08. Copy Cached Ollama Model from Google Drive to Colab
# ==============================================================================
#
# What this cell does:
#   Copies the cached Gemma model files from your Google Drive into the Colab runtime's 
#   local storage (`/root/.ollama/models`).
#
# Execution Time:
#   Takes roughly 2-3 minutes.. Once completed, your model is ready to run at 
#   high GPU speeds.
# ==============================================================================

In [ ]:
# 2. Safely copy the massive brain from Drive to Colab's fast internal disk (~90 seconds)
import os
import shutil

drive_models_dir = "/content/drive/MyDrive/SmartBEM-Studio/ollama_models"
local_models_dir = "/root/.ollama/models"

if os.path.exists(drive_models_dir) and os.listdir(drive_models_dir):
    print("Copying model from Drive to local disk...")
    os.makedirs(local_models_dir, exist_ok=True)
    for item in os.listdir(drive_models_dir):
        s = os.path.join(drive_models_dir, item)
        d = os.path.join(local_models_dir, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    print("Copy complete!")
else:
    print("No cached models found on Google Drive. Will download on-demand.")


## 09. Run Ollama local server

In [ ]:
# ==============================================================================
# ## 09. Run Ollama Local Server
# ==============================================================================
# This step launches the Ollama background daemon service on the GPU runtime.
#
#   1. Sets the network host environment variable to enable standard connections.
#   2. Terminates any stale or frozen background instances of Ollama.
#   3. Launches the Ollama service in the background, logging status outputs 
#      to `/content/ollama_serve.log`.
#   4. Tests server readiness using a local HTTP curl request.
# ==============================================================================

In [ ]:
# 4. Set host environment variables to allow connections
import os
import time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

In [ ]:
# 3. Boot Server in a directory that will never be deleted to avoid getcwd errors on rerun
# First, kill any existing stale ollama instances
!pkill -f ollama
time.sleep(1)

# Move to /content to start the background process safely
%cd /content
!nohup ollama serve > /content/ollama_serve.log 2>&1 &
time.sleep(5)

# Return to the active workspace directory
%cd /content/SmartBEM-Studio/backend_server
print("✅ Ollama Server Awake and ready!")


In [ ]:
# 6. Test if it is running
!curl http://localhost:11434

## 10. Test Ollama server and model up and runnig

In [ ]:
# ==============================================================================
# ## 10. Test Ollama Server and Model
# ==============================================================================
# Verify that the server is active and the Gemma 3 model is loaded correctly.
#
# What to expect:
#   - The run of this cell will take about 2-3 minutes. This is because 
#     Ollama needs to load the model parameters from SSD into the GPU VRAM.
#   - Once loaded, it will print a short response.
#
# How to verify loading success:
#   1. It will print a short response.
#   2. Look at the top right of your Colab screen and click "View resources".
#   3. In the "GPU RAM" tab, you will see a sudden usage spike to 8-9 GB.
#   4. If the cell takes more than 7-8 minutes or returns an error, restart the 
#      session (Runtime -> Restart session) and run again from Step 01.
# ==============================================================================

In [ ]:
"""### Testing on Text Input"""

import ollama
response = ollama.chat(model='gemma3:12b', messages=[
  {'role': 'user', 'content': 'what LLM model are you, answer short'},
])
print(response['message']['content'])

## 11. Install and run EnergyPlus

In [ ]:
# ==============================================================================
# ## 11. Install Python Packages & EnergyPlus Wrapper
# ==============================================================================
# Installs backend server dependencies and the Python EnergyPlus utility wrapper.
#
#   - Installs `energy-plus-utility` from GitHub (developed by @mugalan). This 
#     wrapper acts as the bridge, enabling Python to run EnergyPlus engine 
#     simulations, extract SQL results, and calculate HVAC performance.
# ==============================================================================

In [ ]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

## 12. Run FastAPI server and Ngrok

In [ ]:
# ==============================================================================
# ## 12. Run FastAPI Server & Expose via Ngrok Tunnel
# ==============================================================================
# Starts your backend API and links it to the frontend user interface.
#
# How to use the outputs:
#   1. Run this cell. It will start the server and output two Ngrok URLs.
#   2. Copy the first printed URL (e.g. https://xxxx.ngrok-free.app).
#   3. Go to the Simulation Setup page on the frontend, paste it under 
#      "Step 1: Connect to ngrok Colab Server Connection", and click "Connect".
#   4. You can open the second URL in a new browser tab to view detailed 
#      backend logs of simulations running step-by-step. Refresh the tab 
#      frequently to view real-time updates.
# ==============================================================================

In [ ]:
from core.fastapi_server import start_server
start_server(port=8000)


In [ ]:
# Ensure the directory and log file exist, then tail
!mkdir -p /tmp/smartbem_sim_runs && touch /tmp/smartbem_sim_runs/backend.log



In [ ]:
!tail -n 100 -f /tmp/smartbem_sim_runs/backend.log
